# Experiment 15: Case Study on Generating Examples for an Image Dataset using GANs

**Aim:** Use a **Generative Adversarial Network (DCGAN)** to generate new, realistic-looking images that resemble an existing image dataset, and study the training dynamics.

**Dataset:** Fashion-MNIST (60,000 28x28 greyscale clothing images, 10 classes). Set `DATASET = "mnist"` to use handwritten digits instead.

**How a GAN works**
* **Generator G(z):** turns a random noise vector `z` into an image (transposed convolutions).
* **Discriminator D(x):** CNN classifier that outputs the probability that an image is real.
* They play a minimax game: `min_G max_D  E[log D(x)] + E[log(1 - D(G(z)))]`. D learns to spot fakes while G learns to fool D.

**Case study parts:** (1) training, (2) watching samples improve over epochs, (3) loss curves, (4) latent-space interpolation, (5) using the generated images to **augment** a classifier's training set.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(21); np.random.seed(21)
DATASET, EPOCHS, BATCH, Z_DIM = "fashion_mnist", 15, 128, 100
print("TensorFlow", tf.__version__)

## 1. Load the dataset
Images are scaled to [-1, 1] to match the generator's `tanh` output.

In [ ]:
ds_mod = keras.datasets.fashion_mnist if DATASET == "fashion_mnist" else keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = ds_mod.load_data()
NAMES = (["T-shirt","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"] if DATASET == "fashion_mnist" else [str(i) for i in range(10)])
X = (x_train.astype("float32") - 127.5) / 127.5; X = X[..., None]
train_ds = tf.data.Dataset.from_tensor_slices(X).shuffle(60000).batch(BATCH, drop_remainder=True).prefetch(2)

def show_grid(imgs, rows, cols, title=None, size=1.1):
    imgs = (np.asarray(imgs).squeeze() + 1) / 2
    fig, axes = plt.subplots(rows, cols, figsize=(cols*size, rows*size))
    for a, im in zip(np.ravel(axes), imgs): a.imshow(im, cmap="gray", vmin=0, vmax=1); a.axis("off")
    if title: fig.suptitle(title)
    plt.show()
show_grid(X[:32], 2, 16, "Real training images")

## 2. Generator and Discriminator

In [ ]:
def build_generator():
    return keras.Sequential([
        layers.Input((Z_DIM,)),
        layers.Dense(7*7*128, use_bias=False), layers.BatchNormalization(), layers.LeakyReLU(0.2), layers.Reshape((7, 7, 128)),
        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False), layers.BatchNormalization(), layers.LeakyReLU(0.2),   # 14x14
        layers.Conv2DTranspose(1, 5, strides=2, padding="same", activation="tanh"),                                                     # 28x28
    ], name="generator")

def build_discriminator():
    return keras.Sequential([
        layers.Input((28, 28, 1)),
        layers.Conv2D(64, 5, strides=2, padding="same"), layers.LeakyReLU(0.2), layers.Dropout(0.3),
        layers.Conv2D(128, 5, strides=2, padding="same"), layers.LeakyReLU(0.2), layers.Dropout(0.3),
        layers.Flatten(), layers.Dense(1),
    ], name="discriminator")

G, D = build_generator(), build_discriminator()
G.summary(); D.summary()

## 3. Losses and training step
Binary cross-entropy: D should output 1 for real and 0 for fake; G is rewarded when D outputs 1 for its fakes (non-saturating loss). One-sided label smoothing (real label = 0.9) stabilises training.

In [ ]:
bce = keras.losses.BinaryCrossentropy(from_logits=True)
g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5); d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)

@tf.function
def train_step(real):
    z = tf.random.normal([tf.shape(real)[0], Z_DIM])
    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        fake = G(z, training=True)
        real_out, fake_out = D(real, training=True), D(fake, training=True)
        d_loss = bce(tf.ones_like(real_out) * 0.9, real_out) + bce(tf.zeros_like(fake_out), fake_out)
        g_loss = bce(tf.ones_like(fake_out), fake_out)
    g_opt.apply_gradients(zip(gt.gradient(g_loss, G.trainable_variables), G.trainable_variables))
    d_opt.apply_gradients(zip(dt.gradient(d_loss, D.trainable_variables), D.trainable_variables))
    return g_loss, d_loss, tf.reduce_mean(tf.sigmoid(real_out)), tf.reduce_mean(tf.sigmoid(fake_out))

## 4. Train and watch the generator improve
The same fixed noise vectors are used after every epoch so progress is directly comparable.

In [ ]:
fixed_z = tf.random.normal([16, Z_DIM]); snapshots, hist = {}, []
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    stats = np.array([[float(v) for v in train_step(b)] for b in train_ds]).mean(0); hist.append(stats)
    print(f"epoch {ep:2d}/{EPOCHS}  G loss {stats[0]:.3f}  D loss {stats[1]:.3f}  D(real) {stats[2]:.2f}  D(fake) {stats[3]:.2f}  [{time.time()-t0:.0f}s]")
    if ep in (1, 3, 5, 10, EPOCHS): snapshots[ep] = G(fixed_z, training=False).numpy()
hist = np.array(hist)

In [ ]:
fig, axes = plt.subplots(len(snapshots), 1, figsize=(16, 1.4 * len(snapshots)))
for a, (ep, imgs) in zip(axes, snapshots.items()):
    a.imshow(np.hstack(((imgs.squeeze() + 1) / 2)), cmap="gray"); a.axis("off"); a.set_title(f"epoch {ep}", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(hist[:, 0], label="Generator"); ax[0].plot(hist[:, 1], label="Discriminator"); ax[0].set_title("Losses"); ax[0].legend(); ax[0].set_xlabel("epoch")
ax[1].plot(hist[:, 2], label="D(real)"); ax[1].plot(hist[:, 3], label="D(fake)"); ax[1].set_title("Discriminator outputs"); ax[1].legend(); ax[1].set_xlabel("epoch")
plt.show()

## 5. Final generated samples vs real images

In [ ]:
show_grid(G(tf.random.normal([64, Z_DIM]), training=False).numpy(), 4, 16, "64 newly generated images (never seen in the dataset)")
show_grid(X[np.random.choice(len(X), 64, replace=False)], 4, 16, "64 real images for comparison")

## 6. Latent-space interpolation
Moving smoothly between two noise vectors gives a smooth morph between two generated images, showing the generator learned a continuous image manifold rather than memorising training images.

In [ ]:
rows = []
for _ in range(4):
    z1, z2 = tf.random.normal([1, Z_DIM]), tf.random.normal([1, Z_DIM])
    zs = tf.concat([z1 * (1 - t) + z2 * t for t in np.linspace(0, 1, 12)], 0)
    rows.append(G(zs, training=False).numpy())
show_grid(np.concatenate(rows), 4, 12, "Latent interpolation (each row: z1 -> z2)", size=1.2)

## 7. Using GAN samples for data augmentation
A small classifier is trained on only 1,000 real images, and again on those 1,000 real images **plus 3,000 GAN-generated images**. The generated images need labels, so we label them with a classifier trained on the real subset (pseudo-labels) and keep only confident ones. Results are compared on the full 10,000-image test set.

In [ ]:
def make_clf():
    return keras.Sequential([layers.Input((28, 28, 1)), layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(), layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
                             layers.Flatten(), layers.Dropout(0.3), layers.Dense(10, activation="softmax")])
Xs, Ys = X[:1000], y_train[:1000]; Xt = ((x_test.astype("float32") - 127.5) / 127.5)[..., None]

base = make_clf(); base.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
base.fit(Xs, Ys, epochs=15, batch_size=32, verbose=0); acc_base = base.evaluate(Xt, y_test, verbose=0)[1]

fake = np.concatenate([G(tf.random.normal([500, Z_DIM]), training=False).numpy() for _ in range(6)])
p = base.predict(fake, verbose=0); keep = p.max(1) > 0.9; fake_x, fake_y = fake[keep], p.argmax(1)[keep]
print(f"Kept {keep.sum()} / {len(fake)} confident generated images")

aug = make_clf(); aug.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
aug.fit(np.concatenate([Xs, fake_x]), np.concatenate([Ys, fake_y]), epochs=15, batch_size=32, verbose=0); acc_aug = aug.evaluate(Xt, y_test, verbose=0)[1]
print(f"Classifier trained on 1000 real images            : {acc_base*100:.2f}%")
print(f"Classifier trained on 1000 real + {keep.sum()} GAN images : {acc_aug*100:.2f}%")

## Conclusion
A DCGAN trained on Fashion-MNIST learned to generate new, recognisable clothing items from pure noise: epoch-by-epoch snapshots show noise -> blobs -> sharp garments; losses oscillate as G and D compete (a healthy GAN keeps D(real) high and D(fake) around 0.3-0.5 rather than collapsing to 0). Smooth latent interpolations indicate the model generalises, and the generated images can supplement scarce real data for downstream tasks. Typical GAN issues to watch for are mode collapse and training instability - mitigated here with BatchNorm, LeakyReLU, label smoothing and Adam(beta1=0.5).